# 📝 [제 8편] 로지스틱 회귀 - 합격일까 불합격일까? (분류)

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Logistic Regression (I) - concept • cross-entropy」 (C.pptx = ML_LEC_16), 「Logistic Regression (II) - example」 (ML_LEC_17)

---

## 🎯 학습 목표
1. 분류(classification) 문제에서 sigmoid 함수가 필요한 이유를 설명하고 함숫값을 계산할 수 있다.
2. 손실함수 cross-entropy 가 정답과 예측의 차이를 어떻게 벌점으로 바꾸는지 이해한다.
3. 로지스틱 회귀를 파이썬으로 학습시켜 합격/불합격을 예측하고, 결정 경계의 의미를 해석할 수 있다.

---

## 🧭 지난 시간 복습
* 제 7편: 선형 회귀 y = Wx + b 를 경사하강법으로 학습시켜 **점수(연속적인 숫자)** 를 예측했습니다.
* 오늘은 **합격(1) / 불합격(0)** 처럼 **종류**를 맞히는 **분류** 문제입니다.

[⬅ 이전 : 제 7편 선형 회귀 (3) - 파이썬으로 직접 학습시키기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/07_linear_regression_python.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 9편 XOR 문제 - 직선 하나로는 나눌 수 없다 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/09_xor.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분 함수 (제 2편, 실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad
print('numerical_derivative 준비 완료')

## 📖 개념 다지기 1. 직선만으로는 부족하다

| 공부 시간 x | 2 | 4 | 6 | 8 | 10 | 12 | 14 | 16 | 18 | 20 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 결과 t (불합격 0, 합격 1) | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 |

직선 y = Wx + b 로 이 데이터를 나타내면 x 가 크면 y 가 1 보다 커지고, x 가 작으면 0 보다 작아집니다.
→ "합격할 **확률**" 처럼 **0 과 1 사이의 값**만 내놓는 함수가 필요합니다.

In [ ]:
x_data = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20]).reshape(10, 1)
t_data = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1]).reshape(10, 1)

W_lin, b_lin = np.polyfit(x_data.ravel(), t_data.ravel(), 1)     # 선형 회귀 직선 (최소제곱)
xs = np.linspace(-5, 30, 200)
plt.figure(figsize=(8, 4.5))
plt.scatter(x_data, t_data, s=80, color='royalblue', zorder=5, label='데이터 (0 또는 1)')
plt.plot(xs, W_lin*xs + b_lin, color='orange', lw=2, label='선형 회귀 직선')
plt.axhline(0, color='k', lw=0.8); plt.axhline(1, color='k', lw=0.8, ls='--')
plt.ylim(-0.8, 1.8); plt.xlabel('공부 시간 x'); plt.ylabel('결과'); plt.grid(alpha=0.3); plt.legend()
plt.title('직선은 0 과 1 사이를 벗어난다'); plt.show()

## 📖 개념 다지기 2. sigmoid 함수

> **sigmoid(z) = 1 / (1 + e<sup>−z</sup>)**

* z 가 아무리 커도, 아무리 작아도 sigmoid(z) 는 항상 **0 과 1 사이**
* sigmoid(0) = 1 / (1 + 1) = ( 　　 )

**[연습 1]** 표를 완성하세요. (e<sup>2</sup> ≒ 7.39,  e<sup>−2</sup> ≒ 0.135)

| z | −10 | −2 | 0 | 2 | 10 |
|:-:|:-:|:-:|:-:|:-:|:-:|
| sigmoid(z) | 0.00005 | 1 / (1 + 7.39) ≒ ( 　 ) | ( 　 ) | 1 / (1 + 0.135) ≒ ( 　 ) | 0.99995 |

### 로지스틱 회귀 = 선형 회귀 + sigmoid
```
 입력 x  →  z = Wx + b (선형 회귀)  →  y = sigmoid(z) (0~1 사이 확률)  →  y ≥ 0.5 이면 1 (합격), 아니면 0 (불합격)
```
* sigmoid 값 y 는 **결과가 1 일 확률** 을 뜻합니다. 예) y = 0.83 → 합격할 확률 83%

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-10, -2, 0, 2, 10]:
    print(f'sigmoid({z:3d}) = {sigmoid(z):.5f}')
zs = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 3.8)); plt.plot(zs, sigmoid(zs), color='crimson', lw=2.5)
plt.axhline(0.5, color='gray', ls='--'); plt.axvline(0, color='gray', ls='--')
plt.title('sigmoid(z) = 1 / (1 + e^(−z))'); plt.xlabel('z'); plt.grid(alpha=0.3); plt.show()

## 🎬 애니메이션 1. W 와 b 가 sigmoid 곡선을 어떻게 바꿀까?

y = sigmoid(Wx + b) 에서
* **1단계**: b = −6 고정, W 를 0.1 → 2 로 → 곡선의 **가파른 정도**가 변한다.
* **2단계**: W = 0.5 고정, b 를 −10 → 0 으로 → 곡선이 **좌우로 이동**한다.

y = 0.5 가 되는 점 (Wx + b = 0, 즉 x = −b/W) 이 **합격/불합격의 경계**입니다.

In [ ]:
#@title 🎬 애니메이션 1 실행
params = [(w, -6.0) for w in np.linspace(0.1, 2.0, 40)] + [(0.5, bb) for bb in np.linspace(-10, 0, 40)]
fig, ax = plt.subplots(figsize=(8, 4.8))
xs = np.linspace(-5, 30, 300)
def draw(i):
    W, b = params[i]
    ax.clear()
    ax.scatter(x_data, t_data, s=60, color='royalblue', zorder=5)
    ax.plot(xs, sigmoid(W*xs + b), color='crimson', lw=2.5)
    ax.axhline(0.5, color='gray', ls='--', lw=1)
    xb = -b / W
    ax.axvline(xb, color='green', ls=':', lw=2); ax.text(xb + 0.3, 0.05, f'경계 x = {xb:.1f}', color='green', fontsize=12)
    stage = '1단계 : W 변화 (가파른 정도)' if i < 40 else '2단계 : b 변화 (좌우 이동)'
    ax.set_title(f'{stage}    y = sigmoid({W:.2f}x + ({b:.1f}))', fontsize=13)
    ax.set_xlim(-5, 30); ax.set_ylim(-0.1, 1.1); ax.grid(alpha=0.3); ax.set_xlabel('공부 시간 x')
anim = animation.FuncAnimation(fig, draw, frames=len(params), interval=100)
plt.close(fig)
show_anim(anim)

## 📖 개념 다지기 3. 분류의 손실함수 : cross-entropy

출력 y 가 확률(0~1)이고 정답 t 가 0 또는 1 이므로 오차² 대신 아래 손실함수를 씁니다.

> **E(W, b) = − Σ<sub>i=1</sub><sup>n</sup> { t<sub>i</sub> log y<sub>i</sub> + (1 − t<sub>i</sub>) log(1 − y<sub>i</sub>) }**

어렵게 보이지만 데이터 하나씩 보면 간단합니다. (log 는 자연로그 ln)
* 정답 t = 1 이면 → 벌점 = **− log y**  (y 가 1 에 가까울수록 벌점 0)
* 정답 t = 0 이면 → 벌점 = **− log(1 − y)**  (y 가 0 에 가까울수록 벌점 0)

**[연습 2]** 벌점을 계산하세요. (ln 0.9 ≒ −0.105,  ln 0.1 ≒ −2.303)

| 정답 t | 예측 y | 벌점 | 판단 |
|:-:|:-:|:-:|:-|
| 1 | 0.9 | − ln 0.9 ≒ 0.105 | 잘 맞힘 → 벌점 작음 |
| 1 | 0.1 | − ln 0.1 ≒ ( 　 ) | 틀림 → 벌점 ( 큼 / 작음 ) |
| 0 | 0.1 | − ln(1 − 0.1) ≒ ( 　 ) | |
| 0 | 0.9 | − ln(1 − 0.9) ≒ ( 　 ) | |

> 💡 자신만만하게 틀릴수록(예: 정답 1 인데 y = 0.001) 벌점이 **아주 커집니다**. (− ln 0.001 ≒ 6.9)
> 이 손실함수도 경사하강법으로 W, b 를 갱신합니다:  W ← W − α × ∂E/∂W,  b ← b − α × ∂E/∂b

In [ ]:
ys = np.linspace(0.001, 0.999, 300)
plt.figure(figsize=(7, 4))
plt.plot(ys, -np.log(ys), color='crimson', lw=2, label='정답 t = 1 일 때 벌점  −log y')
plt.plot(ys, -np.log(1 - ys), color='royalblue', lw=2, label='정답 t = 0 일 때 벌점  −log(1 − y)')
plt.xlabel('예측 y'); plt.ylabel('벌점'); plt.ylim(0, 5); plt.legend(); plt.grid(alpha=0.3)
plt.title('cross-entropy 벌점'); plt.show()

## 💻 실습 1. 공부 시간으로 합격/불합격 예측하기

선형 회귀(제 7편)와 달라진 곳은 **딱 두 군데** 입니다. ① y = sigmoid(z) ② 손실함수 = cross-entropy

In [ ]:
np.random.seed(0)
W = np.random.rand(1, 1)
b = np.random.rand(1)

def loss_func(x, t):
    delta = 1e-7                                   # log(0) 이 무한대가 되는 것을 막는 아주 작은 값
    z = np.dot(x, W) + b
    y = sigmoid(z)
    return -np.sum(t*np.log(y + delta) + (1 - t)*np.log((1 - y) + delta))

error_val = loss_func

def predict(x):
    y = sigmoid(np.dot(x, W) + b)                  # 합격할 확률
    result = (y >= 0.5).astype(int)                # 0.5 이상이면 1 (합격)
    return y, result

learning_rate = 1e-2
f = lambda x : loss_func(x_data, t_data)
history = [(0, W[0,0], b[0])]
print('처음 손실 =', error_val(x_data, t_data))
for step in range(10001):
    W -= learning_rate * numerical_derivative(f, W)
    b -= learning_rate * numerical_derivative(f, b)
    if step % 100 == 0:
        history.append((step + 1, W[0,0], b[0]))
    if step % 2000 == 0:
        print(f'step {step:5d} : 손실 = {error_val(x_data, t_data):.5f},  W = {W[0,0]:.4f},  b = {b[0]:.4f}')

In [ ]:
for hours in [3, 17]:
    prob, res = predict(np.array([[hours]]))
    print(f'공부 시간 {hours:2d}시간 → 합격 확률 {prob[0,0]:.4f} → 결과 {res[0,0]} ({"합격" if res[0,0] else "불합격"})')
print('합격/불합격의 경계 x = −b/W =', -b[0]/W[0,0])

> **[확인 1]** 3시간 공부하면 ( 합격 / 불합격 ), 17시간이면 ( 합격 / 불합격 ) 으로 예측한다.
> 경계는 약 ( 　 ) 시간이다. 데이터(12시간은 0, 14시간은 1)와 비교하면 그럴듯한가요?

## 🎬 애니메이션 2. sigmoid 곡선이 학습되는 과정

In [ ]:
#@title 🎬 애니메이션 2 실행 (실습 1 을 먼저 실행하세요)
hist = np.array(history)
sel = list(range(0, 30)) + list(range(30, len(hist), 3)) + [len(hist)-1]*6
fig, ax = plt.subplots(figsize=(8, 4.8))
xs = np.linspace(0, 22, 300)
def draw(f):
    st, w, bb = hist[sel[f]]
    ax.clear()
    ax.scatter(x_data, t_data, s=70, color='royalblue', zorder=5)
    ax.plot(xs, sigmoid(w*xs + bb), color='crimson', lw=2.5)
    ax.axhline(0.5, color='gray', ls='--', lw=1)
    if w > 0:
        ax.axvline(-bb/w, color='green', ls=':', lw=2)
    ax.set_ylim(-0.1, 1.1); ax.set_xlim(0, 22); ax.grid(alpha=0.3); ax.set_xlabel('공부 시간 x')
    ax.set_title(f'step {int(st)} :  y = sigmoid({w:.3f}x + ({bb:.3f}))', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(sel), interval=110)
plt.close(fig)
show_anim(anim)

## 💻 실습 2. 예습 시간과 복습 시간으로 합격 예측하기 (다변수)

| 예습 x₁ | 2 | 4 | 6 | 8 | 10 | 12 | 14 | 16 | 18 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 복습 x₂ | 4 | 11 | 6 | 5 | 7 | 16 | 8 | 3 | 7 |
| 결과 t | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | 1 |

　z = W₁x₁ + W₂x₂ + b,  y = sigmoid(z)  → 　(9 × 2) · (2 × 1) = (9 × 1)

y = 0.5 가 되는 곳, 즉 **W₁x₁ + W₂x₂ + b = 0** 은 평면 위의 **직선** 이 됩니다. 이 직선을 **결정 경계** 라고 하며, 합격과 불합격 영역을 나눕니다.

In [ ]:
x_data = np.array([[2, 4], [4, 11], [6, 6], [8, 5], [10, 7], [12, 16], [14, 8], [16, 3], [18, 7]])
t_data = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1]).reshape(9, 1)

np.random.seed(0)
W = np.random.rand(2, 1)
b = np.random.rand(1)
f = lambda x : loss_func(x_data, t_data)
learning_rate = 1e-2
history2 = [(0, W.copy(), b.copy())]
for step in range(10001):
    W -= learning_rate * numerical_derivative(f, W)
    b -= learning_rate * numerical_derivative(f, b)
    if step % 100 == 0:
        history2.append((step + 1, W.copy(), b.copy()))
    if step % 2000 == 0:
        print(f'step {step:5d} : 손실 = {error_val(x_data, t_data):.5f},  W = {W.ravel()},  b = {b}')

In [ ]:
tests = np.array([[3, 17], [5, 8], [7, 21], [12, 0]])
prob, res = predict(tests)
for (a, r), p, c in zip(tests, prob.ravel(), res.ravel()):
    print(f'예습 {a:2d}시간, 복습 {r:2d}시간 → 합격 확률 {p:.4f} → {"합격" if c else "불합격"}')
print('\nW₁ (예습) =', W[0,0], ',  W₂ (복습) =', W[1,0])

> **[확인 2]** W₁ ( > / < ) W₂ 이므로, 이 데이터에서는 ( 예습 / 복습 ) 시간이 합격에 더 큰 영향을 준다.
> (예습 3, 복습 17) 은 복습을 많이 했는데도 ( 합격 / 불합격 ) 으로 예측되었다. 그 이유는?
> (                                                                                   )

## 🎬 애니메이션 3. 결정 경계가 자리 잡는 과정

배경색은 합격 확률(빨강: 합격 쪽, 파랑: 불합격 쪽), 검은 선은 결정 경계 W₁x₁ + W₂x₂ + b = 0 입니다.

In [ ]:
#@title 🎬 애니메이션 3 실행 (실습 2 를 먼저 실행하세요)
G1, G2 = np.meshgrid(np.linspace(0, 20, 80), np.linspace(0, 20, 80))
grid = np.c_[G1.ravel(), G2.ravel()]
sel = list(range(0, 25)) + list(range(25, len(history2), 3)) + [len(history2)-1]*6
fig, ax = plt.subplots(figsize=(6.5, 6))
def draw(f):
    st, Wk, bk = history2[sel[f]]
    P = sigmoid(grid @ Wk + bk).reshape(G1.shape)
    ax.clear()
    ax.contourf(G1, G2, P, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.6)
    ax.contour(G1, G2, P, levels=[0.5], colors='k', linewidths=2.5)
    for (a, r), tt in zip(x_data, t_data.ravel()):
        ax.scatter(a, r, s=110, marker='o' if tt else 'x', color='darkred' if tt else 'navy', lw=2.5, zorder=5)
    ax.set_xlabel('예습 시간 x1'); ax.set_ylabel('복습 시간 x2'); ax.set_xlim(0, 20); ax.set_ylim(0, 20)
    ax.set_title(f'step {st}   (o 합격, x 불합격)', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(sel), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 3]** 결정 경계는 ( 세로선에 가깝게 / 가로선에 가깝게 ) 기울어져 있다. 이것은 합격 여부가 주로 ( 예습 x₁ / 복습 x₂ ) 에 따라 정해진다는 뜻이다.

## ✏️ 연습 문제

> **[문제 1]** 학습된 W = 1.15, b = −14.86 (실습 1 결과와 비슷한 값) 일 때, 공부 시간 x = 13 시간의 합격 확률을 손으로 어림해 보세요.
> z = 1.15 × 13 − 14.86 = ( 　　 ),  sigmoid(z) ≒ ( 　　 ) → ( 합격 / 불합격 )

> **[문제 2]** W = 1.15, b = −14.86 인 모델에서 1 ~ 20시간 중 **합격 확률이 처음으로 90% 를 넘는 시간**을 찾는 코드를 완성하세요.

In [ ]:
# [문제 2]
W1, b1 = 1.15, -14.86
for hours in range(1, 21):
    prob = sigmoid(W1*hours + b1)
    if prob > ______:
        print(hours, '시간에 처음으로 90% 를 넘음, 확률 =', prob)
        ______                       # 반복을 멈추는 명령어

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
z = 1.15*13 - 14.86
print(f'문제 1 : z = {z:.2f}, sigmoid(z) = {sigmoid(z):.3f} → 0.5 이상이므로 합격 (경계 약 12.9시간 바로 위)')
W1, b1 = 1.15, -14.86
for hours in range(1, 21):
    p = sigmoid(W1*hours + b1)
    if p > 0.9:
        print(f'문제 2 : {hours}시간 (확률 {p:.3f})  ← 빈칸: 0.9, break'); break

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 분류 | 결과가 0 또는 1 인 문제 (합격/불합격, 스팸/정상) |
| 모델 | z = Wx + b → y = sigmoid(z) = 1 / (1 + e<sup>−z</sup>) → y ≥ 0.5 이면 1 |
| 손실함수 | cross-entropy  E = − Σ { t log y + (1 − t) log(1 − y) } |
| 학습 | 선형 회귀와 똑같이 경사하강법 + 수치미분 |
| 결정 경계 | Wx + b = 0 (sigmoid = 0.5 인 곳), 다변수이면 직선(평면) |

### 🏆 도전 과제
로지스틱 회귀의 결정 경계는 항상 **직선** 입니다. 그렇다면 직선 하나로는 나눌 수 없는 데이터도 있을까요?
아래처럼 (0,0)·(1,1) 은 0, (0,1)·(1,0) 은 1 인 데이터를 직선 하나로 나눠 보세요. → 다음 시간: **XOR 문제**

(                                                                                                                            )

---

[⬅ 이전 : 제 7편 선형 회귀 (3) - 파이썬으로 직접 학습시키기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/07_linear_regression_python.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 9편 XOR 문제 - 직선 하나로는 나눌 수 없다 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/09_xor.ipynb)